## 1. Stereo Calibration
Camera Intrinsics and Extrinsics!

In [ ]:
import sys
from pathlib import Path

sys.path[:0] = [str(Path.cwd() / d) for d in ("calib", "camera", "pose", "viz")]
from calibrate import PAIR_DIR, SPEC, run_calibration
from capture import capture
from plots import undistort_figure
from results import write_results
from elp import probe_indices

probe_indices()

In [ ]:
capture(PAIR_DIR, indices=(0, 1), spec=SPEC)     # reuses the bag if it already exists
# capture(PAIR_DIR, override=True)               # re-shoot it; SPACE pauses, q quits

In [ ]:
cal = run_calibration(SPEC, PAIR_DIR)
write_results(cal, SPEC) if cal["passed"] else print("gate failed, nothing written")
undistort_figure(cal)                            # straight edges should straighten

In [ ]:
# !python calib/calibrate.py                     # capture (or reuse the bag), solve, write
!python calib/calibrate.py --no-capture      # re-solve the bag already on disk
# !python calib/calibrate.py --override        # re-shoot the bag, replacing it

## 2. Record

Capture and save stereo camera recording as mp4

In [ ]:
from record import DEFAULT_DIR, latest_flight, record
# record(DEFAULT_DIR, indices=(0, 1))              # SPACE starts and stops a flight,
                                                 # each take its own dated folder

## 3. Visualise

5 DOF pose estimation viser visualisation

Optional. Both paths now build their own plate from the stream as it runs (`background.RunningPlate`), so nothing has to be shot in advance and a stale plate cannot be picked up by mistake. Shoot one anyway if you prefer -- **take the robot out of frame**, run the cell below once, put it back -- and `from_stereo` will use it.

In [ ]:
import background

background.capture_stereo()          # optional; writes pose/background_A.png, _B.png

In [ ]:
from live_viz import from_recording, from_stereo, replay

from_stereo("camera:0,camera:1")                 # live; interrupt to stop
#   no rotate: the datum already puts the rotor axis on +z and the scene follows it.
#   rotate=("y", 90) turns the frame right-handed about +y, moving up to +x; the
#   datum line printed at startup says where up ended up.
#   backgrounds="running" builds the plate from the stream; "saved" uses the cell above
# flight = latest_flight(DEFAULT_DIR)                             # or pick one
# from_recording(flight, csv_out=flight / "poses.csv")            # offline, from mp4
# replay(flight / "poses.csv")                                    # offline, from the CSV